# CS3081: Uncertainty & Bayesian Networks
## Exercise 5 (c): pgmpy Implementation

**Student Name:** Ghaida Alsaadi | **ID:** S21107336

**Network Topology:** `heatwave → overload`, `heatwave → powercut`, `overload → powercut`, `powercut → cancel`.

### Step 0: Environment Setup

In [1]:
!pip install pgmpy -q


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\Admin.DESKTOP-QKNRRPJ\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Step 1: Define the Bayesian Network and CPDs

In [2]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD, State
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling

# 1. Define network edges
edges = [
    ("heatwave", "overload"),
    ("heatwave", "powercut"),
    ("overload", "powercut"),
    ("powercut", "cancel")
]
bn_model = DiscreteBayesianNetwork(edges)

states = ["yes", "no"]

# 2. Define Conditional Probability Distributions (CPDs)
p_heatwave = TabularCPD(
    variable="heatwave", variable_card=2,
    values=[[0.3], [0.7]],
    state_names={"heatwave": states}
)

p_overload = TabularCPD(
    variable="overload", variable_card=2,
    values=[[0.6, 0.1],
            [0.4, 0.9]],
    evidence=["heatwave"], evidence_card=[2],
    state_names={"overload": states, "heatwave": states}
)

p_powercut = TabularCPD(
    variable="powercut", variable_card=2,
    values=[[0.7, 0.2, 0.5, 0.05],
            [0.3, 0.8, 0.5, 0.95]],
    evidence=["heatwave", "overload"], evidence_card=[2, 2],
    state_names={"powercut": states, "heatwave": states, "overload": states}
)

p_cancel = TabularCPD(
    variable="cancel", variable_card=2,
    values=[[0.8, 0.1],
            [0.2, 0.9]],
    evidence=["powercut"], evidence_card=[2],
    state_names={"cancel": states, "powercut": states}
)

# 3. Add CPDs to the model and verify
bn_model.add_cpds(p_heatwave, p_overload, p_powercut, p_cancel)
print("Model validation check:", bn_model.check_model())

C:\Users\Admin.DESKTOP-QKNRRPJ\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Admin.DESKTOP-QKNRRPJ\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Model validation check: True


### Step 2: Exact Inference using Variable Elimination

In [3]:
ve_solver = VariableElimination(bn_model)
query_res = ve_solver.query(variables=["heatwave"], evidence={"cancel": "yes"}, show_progress=False)
print(query_res)

exact_prob = query_res.get_value(heatwave="yes")
print(f"-> Exact Probability P(heatwave=yes | cancel=yes) = {exact_prob:.4f}")

+---------------+-----------------+
| heatwave      |   phi(heatwave) |
+===============+=================+
| heatwave(yes) |          0.5367 |
+---------------+-----------------+
| heatwave(no)  |          0.4633 |
+---------------+-----------------+
-> Exact Probability P(heatwave=yes | cancel=yes) = 0.5367


### Step 3: Approximate Inference (Rejection Sampling)

In [4]:
n_samples = 10000  # Assignment requirement
sampler_obj = BayesianModelSampling(bn_model)

rej_samples = sampler_obj.rejection_sample(
    evidence=[State("cancel", "yes")],
    size=n_samples,
    show_progress=False
)

print(f"Total valid samples generated: {len(rej_samples)}")
print(rej_samples.head())

approx_prob = (rej_samples["heatwave"] == "yes").mean()
print(f"\n-> Estimated P(heatwave=yes | cancel=yes) = {approx_prob:.4f}")
print(f"-> Absolute Error = {abs(approx_prob - exact_prob):.4f}")

Total valid samples generated: 10000
  heatwave overload powercut cancel
0       no      yes      yes    yes
1      yes      yes      yes    yes
2      yes      yes      yes    yes
3      yes      yes      yes    yes
4       no       no      yes    yes

-> Estimated P(heatwave=yes | cancel=yes) = 0.5354
-> Absolute Error = 0.0013


### Step 4: Final Conclusion
The exact probability calculated via Variable Elimination is `0.5367`. Using Rejection Sampling with 10,000 iterations, we get an estimated value that is very close to the exact probability. The minor difference represents the expected variance in random sampling methods, as required by the sampling approximation method.